# Test FIR Filter bằng AXI DMA trên PYNQ-Z2

Notebook này load overlay, tạo tín hiệu test, truyền qua AXI DMA/FIR, nhận output và vẽ sóng/FFT.

## Cell 1 — Import thư viện

`Overlay` load bitstream, `allocate` cấp buffer DMA, `numpy` xử lý tín hiệu, `matplotlib` vẽ đồ thị.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully")


## Cell 2 — Load overlay

Input: `fir_filter_ip_2023_1.bit` và `fir_filter_ip_2023_1.hwh` cùng thư mục notebook. Output: danh sách IP, cần thấy `axi_dma_0`.

In [ ]:
overlay = Overlay("fir_filter_ip_2023_1.bit")
overlay.download()

print("Overlay loaded successfully")
print("IP blocks found in overlay:")
for ip_name in overlay.ip_dict.keys():
    print(" -", ip_name)


## Cell 3 — Lấy AXI DMA

`sendchannel` = MM2S, gửi DDR → Stream. `recvchannel` = S2MM, nhận Stream → DDR.

In [ ]:
dma = overlay.axi_dma_0

print("DMA object:", dma)
print("Send channel:", dma.sendchannel)
print("Recv channel:", dma.recvchannel)


## Cell 4 — Tạo tín hiệu test

Tín hiệu gồm 1 kHz và 10 kHz, sau đó scale sang `int32` để đưa vào FIR Compiler 32-bit signed.

In [ ]:
Fs = 48000
N = 1024

f_low = 1000
f_high = 10000

n = np.arange(N)

x_float = (
    0.7 * np.sin(2 * np.pi * f_low * n / Fs)
    + 0.3 * np.sin(2 * np.pi * f_high * n / Fs)
)

scale = 2**20
x_int = np.round(x_float * scale).astype(np.int32)

print("Number of samples:", N)
print("Sampling frequency:", Fs, "Hz")
print("Input min:", x_int.min())
print("Input max:", x_int.max())
print("First 16 input samples:")
print(x_int[:16])


## Cell 5 — Cấp buffer DMA

DMA cần buffer có địa chỉ vật lý trong DDR. `allocate()` tạo buffer phù hợp.

In [ ]:
input_buffer = allocate(shape=(N,), dtype=np.int32)
output_buffer = allocate(shape=(N,), dtype=np.int32)

input_buffer[:] = x_int
output_buffer[:] = 0

print("Input buffer physical address :", hex(input_buffer.physical_address))
print("Output buffer physical address:", hex(output_buffer.physical_address))
print("Input buffer shape:", input_buffer.shape)
print("Output buffer shape:", output_buffer.shape)


## Cell 6 — Chạy DMA qua FIR

Start `recvchannel` trước, rồi start `sendchannel`. Sau khi xong, invalidate output để CPU đọc dữ liệu mới.

In [ ]:
input_buffer.flush()
output_buffer.flush()

dma.recvchannel.transfer(output_buffer)
dma.sendchannel.transfer(input_buffer)

dma.sendchannel.wait()
dma.recvchannel.wait()

output_buffer.invalidate()

y_int = np.array(output_buffer, dtype=np.int32)

print("DMA transfer completed")
print("Output min:", y_int.min())
print("Output max:", y_int.max())
print("First 16 output samples:")
print(y_int[:16])


## Cell 7 — Vẽ input/output miền thời gian

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(x_int[:300], label="Input x[n]")
plt.plot(y_int[:300], label="Output y[n] after FIR")
plt.title("FIR Filter Input and Output - Time Domain")
plt.xlabel("Sample index n")
plt.ylabel("Amplitude")
plt.grid(True)
plt.legend()
plt.show()


## Cell 8 — Chuẩn hóa để dễ nhìn

Do hệ số FIR chưa normalize, output có thể lớn hơn input.

In [ ]:
x_norm = x_int.astype(np.float64)
y_norm = y_int.astype(np.float64)

x_norm = x_norm / np.max(np.abs(x_norm))
y_norm = y_norm / np.max(np.abs(y_norm))

plt.figure(figsize=(14, 5))
plt.plot(x_norm[:300], label="Normalized input")
plt.plot(y_norm[:300], label="Normalized output")
plt.title("Normalized Input and Output")
plt.xlabel("Sample index n")
plt.ylabel("Normalized amplitude")
plt.grid(True)
plt.legend()
plt.show()


## Cell 9 — FFT trước/sau lọc

FFT giúp xem phổ input và output.

In [ ]:
freq = np.fft.rfftfreq(N, d=1/Fs)

X = np.abs(np.fft.rfft(x_int))
Y = np.abs(np.fft.rfft(y_int))

plt.figure(figsize=(14, 5))
plt.plot(freq, X, label="Input FFT")
plt.plot(freq, Y, label="Output FFT")
plt.title("FFT Before and After FIR Filter")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.grid(True)
plt.legend()
plt.xlim(0, 20000)
plt.show()


## Cell 10 — Đo gain tại 1 kHz và 10 kHz

In [ ]:
def nearest_bin(freq_array, target_freq):
    return np.argmin(np.abs(freq_array - target_freq))

idx_1k = nearest_bin(freq, 1000)
idx_10k = nearest_bin(freq, 10000)

gain_1k = Y[idx_1k] / X[idx_1k]
gain_10k = Y[idx_10k] / X[idx_10k]

print("Nearest bin for 1 kHz :", freq[idx_1k], "Hz")
print("Nearest bin for 10 kHz:", freq[idx_10k], "Hz")
print("Gain at 1 kHz :", gain_1k)
print("Gain at 10 kHz:", gain_10k)


## Cell 11 — Test nhanh bằng ramp

Nếu cell này treo, kiểm tra TLAST/TREADY/AXI4-Stream.

In [ ]:
N2 = 64

input_ramp = allocate(shape=(N2,), dtype=np.int32)
output_ramp = allocate(shape=(N2,), dtype=np.int32)

input_ramp[:] = np.arange(N2, dtype=np.int32)
output_ramp[:] = 0

input_ramp.flush()
output_ramp.flush()

dma.recvchannel.transfer(output_ramp)
dma.sendchannel.transfer(input_ramp)

dma.sendchannel.wait()
dma.recvchannel.wait()

output_ramp.invalidate()

print("Input ramp:")
print(np.array(input_ramp))

print("Output ramp:")
print(np.array(output_ramp))


## Cell 12 — Giải phóng buffer

In [ ]:
input_buffer.freebuffer()
output_buffer.freebuffer()

try:
    input_ramp.freebuffer()
    output_ramp.freebuffer()
except Exception:
    pass

print("Buffers released")
